In [2]:

from google.colab import files
uploaded = files.upload()

Saving test.tsv to test.tsv
Saving train.tsv to train.tsv
Saving valid.tsv to valid.tsv


In [7]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# -------------------------
# LOAD DATA
# -------------------------
columns = [
    "id", "label", "statement", "subject", "speaker",
    "job_title", "state_info", "party_affiliation",
    "barely_true", "false", "half_true", "mostly_true",
    "pants_on_fire", "context"
]

df = pd.read_csv("train.tsv", sep="\t", names=columns)

# -------------------------
# FEATURE ENGINEERING
# -------------------------
df["text"] = df["statement"].fillna("") + " " + df["subject"].fillna("")

X = df["text"]
y = df["label"]

# -------------------------
# TRAIN-TEST SPLIT
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# MODEL PIPELINE
# -------------------------
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        max_features=12000,
        ngram_range=(1,2)
    )),
    ("clf", LogisticRegression(
        max_iter=500,
        class_weight="balanced"
    ))
])

# -------------------------
# TRAIN MODEL
# -------------------------
model.fit(X_train, y_train)

# -------------------------
# PREDICT
# -------------------------
y_pred = model.predict(X_test)

# -------------------------
# EVALUATION
# -------------------------
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# -------------------------
# SAVE MODEL
# -------------------------
joblib.dump(model, "liar_model.pkl")

print("Model saved successfully!")

Accuracy: 0.23291015625
              precision    recall  f1-score   support

 barely-true       0.21      0.23      0.22       331
       false       0.27      0.24      0.26       399
   half-true       0.22      0.16      0.19       423
 mostly-true       0.25      0.26      0.25       392
  pants-fire       0.19      0.29      0.23       168
        true       0.24      0.26      0.25       335

    accuracy                           0.23      2048
   macro avg       0.23      0.24      0.23      2048
weighted avg       0.24      0.23      0.23      2048

Model saved successfully!


In [8]:
from google.colab import files
files.download("liar_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
!pip install streamlit pyngrok joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 106.1 MB/s eta 0:00:00


In [11]:
%%writefile app.py

import streamlit as st
import joblib

model = joblib.load("liar_model.pkl")

st.title("📰 LIAR Fake News Detection")

user_input = st.text_area("Enter statement")

if st.button("Predict"):
    if user_input:
        pred = model.predict([user_input])[0]
        st.success(f"Prediction: {pred}")
    else:
        st.warning("Enter text first")

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-05-14 14:08:41.371 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.65.133:8501



In [ ]:
from pyngrok import ngrok

In [ ]:
import streamlit as st
import joblib

# Load model
model = joblib.load("model/liar_model.pkl")

st.title("📰 LIAR Fake News Detection App")

user_input = st.text_area("Enter a statement")

if st.button("Predict"):
    if user_input:
        pred = model.predict([user_input])[0]
        st.success(f"Prediction: {pred}")
    else:
        st.warning("Please enter text")